# Recent Analysis
Analyze last 12 months of vulnerability data with incremental updates.

In [1]:
from vulnflow.helpers import CacheBuilder

ds = CacheBuilder(years=[2024, 2025], include_modified=True).build()
df = ds.to_pandas()
print(f"{len(df):,} CVEs  |  {len(df.columns)} columns")

Extracting NVD Data:   0%|          | 0.00/3.00 [00:00]

Parsing nvdcve-2.0-Modified:   0%|          | 0.00/2.31k [00:00<?, ?CVEs/s]

Parsing nvdcve-2.0-2025:   0%|          | 0.00/42.3k [00:00<?, ?CVEs/s]

Parsing nvdcve-2.0-2024:   0%|          | 0.00/39.0k [00:00<?, ?CVEs/s]

81,162 CVEs  |  23 columns


## Severity Overview

In [2]:
import pandas as pd

severity = df["cvss_base_severity"].value_counts()
total_scored = severity.sum()
print(f"{total_scored:,} CVEs with CVSS scores\n")
severity

77,679 CVEs with CVSS scores



cvss_base_severity
MEDIUM      37387
HIGH        27773
CRITICAL    10677
LOW          1836
NONE            6
Name: count, dtype: int64

## Monthly Trends

In [3]:
df["published_month"] = pd.to_datetime(df["nvd_published_at"]).dt.to_period("M")
monthly = df.groupby("published_month").size()
monthly.tail(12)

published_month
2025-03    3876
2025-04    3975
2025-05    3670
2025-06    3358
2025-07    3723
2025-08    3458
2025-09    3816
2025-10    3915
2025-11    3026
2025-12    4541
2026-01    2687
2026-02    1509
Freq: M, dtype: int64

## CVSS Score Distribution

In [4]:
scores = df["cvss_base_score"].dropna()
print(f"Mean: {scores.mean():.1f}  |  Median: {scores.median():.1f}  |  Std: {scores.std():.1f}")
print(f"Range: {scores.min():.1f} - {scores.max():.1f}")

Mean: 6.9  |  Median: 6.8  |  Std: 1.7
Range: 0.0 - 10.0


## Attack Vector Breakdown

In [5]:
df["cvss_attack_vector"].value_counts()

cvss_attack_vector
N    59129
L    16040
A     1843
P      667
Name: count, dtype: int64

## Save & Incremental Update

In [6]:
ds.to_parquet("../data/recent_cache.parquet")
print("Saved to data/recent_cache.parquet")

Saved to data/recent_cache.parquet


In [7]:
# Incremental update: merge existing cache with latest Modified feed
from vulnflow import VulnDataset
from vulnflow.extractors.nvd import NVDExtractor

cache = VulnDataset.from_parquet("../data/recent_cache.parquet")
delta = NVDExtractor(CacheBuilder.feed_url("modified")).extract()
updated = VulnDataset.merge([cache, delta])
updated.to_parquet("../data/recent_cache.parquet")
print(f"Cache: {len(cache):,} → Updated: {len(updated):,} CVEs")

Parsing nvdcve-2.0-Modified:   0%|          | 0.00/2.31k [00:00<?, ?CVEs/s]

Cache: 83,610 → Updated: 85,922 CVEs
